# 🚀 StyleVision AI — Unified Multi-Studio Cloud GPU Server
### High-Fidelity Neural Virtual Try-On on Free NVIDIA T4 GPU (15 GB VRAM)

This unified Google Colab server hosts **Neural Diffusion Models** with zero-overhead memory swapping:

| Studio | AI Model | Role |
| :--- | :--- | :--- |
| **👗 Haute Wardrobe** | **CatVTON** | Draping gowns, sarees, jackets, dresses, and tops |
| **👜 Bags & Accessories** | **Inpainting Studio** | Zero-shot localized inpainting for handbags, hats, shoes |
| **💎 Fine Jewelry & Eyewear**| **IP-Adapter Inpainting** | Photorealistic lighting & skin reflection for necklaces, earrings, sunglasses |

---
### ⚙️ Step 1: Ensure GPU is Enabled
Go to **Runtime > Change runtime type > Hardware accelerator > T4 GPU** before running!

In [ ]:
# @title 1. Verify NVIDIA GPU Acceleration
!nvidia-smi

import torch
print('='*60)
if torch.cuda.is_available():
    print(f'✅ GPU Detected: {torch.cuda.get_device_name(0)}')
    print(f'⚡ Dedicated VRAM: {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB')
else:
    print('❌ No GPU detected! Go to Runtime -> Change runtime type -> Select T4 GPU.')
print('='*60)

### 📦 Step 2: Install Unified Dependencies (~2 mins)
Installs `diffusers`, `transformers`, `accelerate`, `fvcore`, `iopath`, `av`, `pycocotools`, and the `cloudflared` tunnel.

In [ ]:
# @title 2. Clone Repositories & Install Dependencies (~2 mins)
import os, sys

work_dir = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
catvton_dir = os.path.join(work_dir, 'CatVTON')
if not os.path.exists(catvton_dir):
    print('📦 Cloning CatVTON...')
    !git clone https://github.com/Zheng-Chong/CatVTON.git {catvton_dir}

if catvton_dir not in sys.path:
    sys.path.insert(0, catvton_dir)
os.chdir(catvton_dir)

print('📦 Installing ML Libraries & Accelerators...')
!pip install -q 'huggingface_hub<0.26.0' 'diffusers==0.27.2' accelerate==0.28.0 transformers==4.39.1 torchvision
!pip install -q fvcore iopath omegaconf av pycocotools cloudpickle
!pip install -q fastapi==0.110.0 uvicorn==0.28.0 python-multipart nest_asyncio
!pip install -q opencv-python-headless

print('🌐 Installing Cloudflare Tunnel (cloudflared)...')
!wget -nc -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print('✅ Dependencies installed! Proceed to Step 3.')

### 📥 Step 3: Load Model Weights into GPU Memory (~2-3 mins)
- **CatVTON**: runwayml/stable-diffusion-inpainting + zhengchong/CatVTON attention & AutoMasker (DensePose + SCHP)
- **IP-Adapter**: Inpainting pipeline with CLIP Image Encoder for fine jewelry, bags, and eyewear

In [ ]:
# @title 3. Download Model Weights & Initialize Pipelines (~2-3 mins)
import sys, types, os, gc, io
from PIL import Image

work_dir = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
catvton_dir = os.path.join(work_dir, 'CatVTON')
if catvton_dir not in sys.path:
    sys.path.insert(0, catvton_dir)
os.chdir(catvton_dir)

import numpy as np

# Compatibility shim: restore numpy.core for cv2 compiled against NumPy 1.x
if not hasattr(np, 'core') or not hasattr(getattr(np, 'core', None), 'multiarray'):
    _core_mod = types.ModuleType('numpy.core')
    _core_mod.multiarray = np._core.multiarray
    _core_mod.umath = np._core.umath
    _core_mod.numeric = np._core.numeric
    np.core = _core_mod
    sys.modules['numpy.core'] = _core_mod
    sys.modules['numpy.core.multiarray'] = np._core.multiarray
    sys.modules['numpy.core.umath'] = np._core.umath
    sys.modules['numpy.core.numeric'] = np._core.numeric
    print('✅ NumPy compatibility shim applied.')

# Compatibility patches for accelerate & peft
try:
    import accelerate.utils
    import accelerate.utils.memory
    def _clear_device_cache():
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    accelerate.utils.memory.clear_device_cache = _clear_device_cache
    accelerate.utils.clear_device_cache = _clear_device_cache
except Exception:
    pass

try:
    import transformers
    if not hasattr(transformers, 'EncoderDecoderCache'):
        class EncoderDecoderCache:
            pass
        transformers.EncoderDecoderCache = EncoderDecoderCache
except Exception:
    pass

import torch
from huggingface_hub import snapshot_download
from diffusers.image_processor import VaeImageProcessor
from model.cloth_masker import AutoMasker
from model.pipeline import CatVTONPipeline
from utils import init_weight_dtype, resize_and_crop, resize_and_padding

print('📥 Step 3A: Downloading & Initializing CatVTON (Clothing Try-On)...')
repo_path = snapshot_download(repo_id='zhengchong/CatVTON')

pipeline = CatVTONPipeline(
    base_ckpt='runwayml/stable-diffusion-inpainting',
    attn_ckpt=repo_path,
    attn_ckpt_version='mix',
    weight_dtype=torch.float16,
    use_tf32=True,
    device='cuda',
    skip_safety_check=True
)
catvton_pipeline = pipeline

print('⚡ Initializing AutoMasker (DensePose + SCHP)...')
mask_processor = VaeImageProcessor(vae_scale_factor=8, do_normalize=False, do_binarize=True, do_convert_grayscale=True)
automasker = AutoMasker(
    densepose_ckpt=os.path.join(repo_path, 'DensePose'),
    schp_ckpt=os.path.join(repo_path, 'SCHP'),
    device='cuda'
)

print('📥 Step 3B: Loading IP-Adapter Inpainting (Jewelry, Bags & Eyewear)...')
from diffusers import StableDiffusionInpaintPipeline
from transformers import CLIPVisionModelWithProjection
try:
    print('  Loading CLIP Vision Encoder...')
    image_encoder = CLIPVisionModelWithProjection.from_pretrained(
        'h94/IP-Adapter',
        subfolder='models/image_encoder',
        torch_dtype=torch.float16,
    )
    print('  Loading Inpainting Pipeline with Vision Adapter...')
    ip_pipe = StableDiffusionInpaintPipeline.from_pretrained(
        'runwayml/stable-diffusion-inpainting',
        image_encoder=image_encoder,
        torch_dtype=torch.float16,
        safety_checker=None
    )
    ip_pipe.load_ip_adapter('h94/IP-Adapter', subfolder='models', weight_name='ip-adapter_sd15.bin')
    ip_pipe.set_ip_adapter_scale(0.75)
    # Kept in CPU memory (~0 MB VRAM) and swapped to CUDA on-demand
    ip_pipe.to('cpu')
    print('✅ IP-Adapter loaded successfully (ready in CPU memory, swaps to CUDA on demand)!')
except Exception as e:
    import traceback
    traceback.print_exc()
    print('⚠️ IP-Adapter notice:', e)
    ip_pipe = None

print('='*60)
print('🎉 All Models Configured Successfully!')
print(f'⚡ Active GPU VRAM: {round(torch.cuda.memory_allocated() / (1024**3), 2)} GB / 15.0 GB')
print('='*60)

### 🌐 Step 4: Launch Unified FastAPI Server + Cloudflare Tunnel
Exposes `/tryon` (CatVTON), `/tryon-jewelry` (IP-Adapter), and `/tryon-accessory` publicly through Cloudflare.

In [ ]:
# @title 4. Start Unified Server + Cloudflare Tunnel (Keep this cell running!)
import os, sys, io, re, socket, subprocess, threading, time, gc
import numpy as np
import torch
import uvicorn
from PIL import Image, ImageDraw
from utils import resize_and_crop, resize_and_padding
from fastapi import FastAPI, File, UploadFile, Form, HTTPException
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware

# Compatibility patches for transformers & accelerate
try:
    import transformers
    if not hasattr(transformers, 'EncoderDecoderCache'):
        class EncoderDecoderCache:
            pass
        transformers.EncoderDecoderCache = EncoderDecoderCache
except Exception:
    pass

try:
    import accelerate.utils
    import accelerate.utils.memory
    def _clear_device_cache():
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    if not hasattr(accelerate.utils.memory, 'clear_device_cache'):
        accelerate.utils.memory.clear_device_cache = _clear_device_cache
    if not hasattr(accelerate.utils, 'clear_device_cache'):
        accelerate.utils.clear_device_cache = _clear_device_cache
except Exception:
    pass

app = FastAPI(title='StyleVision Unified AI GPU Studio')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])

@app.get('/')
@app.get('/health')
def health():
    vram = round(torch.cuda.memory_allocated() / (1024**3), 2) if torch.cuda.is_available() else 0.0
    ip_status = 'available' if ('ip_pipe' in globals() and ip_pipe is not None) else 'offline'
    return {
        'status': 'online',
        'service': 'StyleVision Unified GPU Server',
        'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'vram_allocated_gb': vram,
        'studios': {
            'catvton': 'available',
            'ip_adapter_jewelry': ip_status,
            'accessory_inpainting': ip_status,
            'hairstyle_inpainting': ip_status
        }
    }

# ─── 1. CatVTON (Clothing Try-On) ──────────────────────────────────────────
@app.post('/tryon')
async def tryon_clothing(
    person_image: UploadFile = File(..., description='Full person photo (pose is preserved)'),
    cloth_image: UploadFile = File(..., description='Target clothing image (saree, dress, gown, top)'),
    cloth_type: str = Form('overall', description='"overall" | "upper" | "lower"'),
    num_steps: int = Form(40, description='Inference steps (30-50)'),
    guidance_scale: float = Form(2.5, description='CFG scale'),
    seed: int = Form(42, description='Seed for reproducibility')
):
    try:
        if 'automasker' not in globals() or ('pipeline' not in globals() and 'catvton_pipeline' not in globals()):
            raise HTTPException(status_code=503, detail='Models not loaded. Please run Cell 3 first.')

        print(f'👗 CatVTON Try-On Request received (cloth_type={cloth_type}, steps={num_steps})...')
        p_bytes = await person_image.read()
        c_bytes = await cloth_image.read()
        person_pil = Image.open(io.BytesIO(p_bytes)).convert('RGB')
        cloth_pil = Image.open(io.BytesIO(c_bytes)).convert('RGB')

        target_w, target_h = 768, 1024
        person_resized = resize_and_crop(person_pil, (target_w, target_h))
        cloth_resized = resize_and_padding(cloth_pil, (target_w, target_h))

        mask = automasker(person_resized, cloth_type)['mask']
        mask = mask_processor.blur(mask, blur_factor=9)

        pipe = globals().get('pipeline', globals().get('catvton_pipeline'))
        generator = torch.Generator(device='cuda').manual_seed(seed) if seed != -1 else None
        with torch.inference_mode():
            result = pipe(
                image=person_resized,
                condition_image=cloth_resized,
                mask=mask,
                num_inference_steps=num_steps,
                guidance_scale=guidance_scale,
                generator=generator
            )[0]

        print('✅ CatVTON inference completed!')
        buf = io.BytesIO()
        result.save(buf, format='PNG')
        buf.seek(0)
        return StreamingResponse(buf, media_type='image/png', headers={'X-TryOn-Status': 'success', 'X-TryOn-Studio': 'CatVTON'})
    except Exception as e:
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

# ─── 2. IP-Adapter (Fine Jewelry & Eyewear) ─────────────────────────────────
@app.post('/tryon-jewelry')
async def tryon_jewelry(
    person_image: UploadFile = File(...),
    accessory_image: UploadFile = File(...),
    subcategory: str = Form('necklace')
):
    try:
        p_bytes = await person_image.read()
        a_bytes = await accessory_image.read()
        person_pil = Image.open(io.BytesIO(p_bytes)).convert('RGB')
        acc_pil = Image.open(io.BytesIO(a_bytes)).convert('RGB')

        w, h = person_pil.size
        mask = Image.new('L', (w, h), 0)
        draw = ImageDraw.Draw(mask)
        sub = (subcategory or 'necklace').lower()

        if 'neck' in sub or 'chain' in sub or 'pendant' in sub:
            draw.rectangle([int(w * 0.28), int(h * 0.28), int(w * 0.72), int(h * 0.48)], fill=255)
            prompt = 'high quality luxury gold necklace, diamond jewelry detail, realistic lighting, highly detailed'
        elif 'glass' in sub or 'shade' in sub or 'optic' in sub:
            draw.rectangle([int(w * 0.25), int(h * 0.16), int(w * 0.75), int(h * 0.28)], fill=255)
            prompt = 'stylish designer eyewear, sunglasses, sharp reflection, photorealistic'
        elif 'ear' in sub:
            draw.rectangle([int(w * 0.18), int(h * 0.20), int(w * 0.32), int(h * 0.32)], fill=255)
            draw.rectangle([int(w * 0.68), int(h * 0.20), int(w * 0.82), int(h * 0.32)], fill=255)
            prompt = 'luxury diamond earrings, glistening jewelry, detailed earlobes'
        elif 'watch' in sub or 'wrist' in sub or 'bracelet' in sub:
            draw.rectangle([int(w * 0.15), int(h * 0.60), int(w * 0.35), int(h * 0.75)], fill=255)
            prompt = 'luxury gold watch bracelet on wrist, elegant timepiece'
        elif 'ring' in sub:
            draw.rectangle([int(w * 0.20), int(h * 0.65), int(w * 0.40), int(h * 0.80)], fill=255)
            prompt = 'luxury sparkling diamond ring on finger, exquisite jewelry'
        else:
            draw.rectangle([int(w * 0.25), int(h * 0.25), int(w * 0.75), int(h * 0.55)], fill=255)
            prompt = f'high quality {sub} accessory, luxury fashion item'

        if 'ip_pipe' in globals() and ip_pipe is not None:
            print(f'💎 Running IP-Adapter diffusion for {sub}...')
            ip_pipe.to('cuda')
            with torch.inference_mode():
                res = ip_pipe(
                    prompt=prompt,
                    image=person_pil.resize((512, 512)),
                    mask_image=mask.resize((512, 512)),
                    ip_adapter_image=acc_pil.resize((224, 224)),
                    num_inference_steps=25,
                    guidance_scale=7.0
                ).images[0]
            ip_pipe.to('cpu')
            torch.cuda.empty_cache()
            out_img = res.resize((w, h))
            print(f'✅ IP-Adapter jewelry try-on completed for {sub}!')
        else:
            print(f'⚠️ IP-Adapter offline: returning baseline image for {sub}')
            out_img = person_pil

        buf = io.BytesIO()
        out_img.save(buf, format='PNG')
        buf.seek(0)
        return StreamingResponse(buf, media_type='image/png', headers={'X-TryOn-Studio': 'IP-Adapter'})
    except Exception as e:
        import traceback
        traceback.print_exc()
        if 'ip_pipe' in globals() and ip_pipe is not None:
            try:
                ip_pipe.to('cpu')
            except Exception:
                pass
            torch.cuda.empty_cache()
        raise HTTPException(status_code=500, detail=str(e))

# ─── 3. Accessories (Bags, Shoes, Hats) ─────────────────────────────────────
@app.post('/tryon-accessory')
async def tryon_accessory(
    person_image: UploadFile = File(...),
    accessory_image: UploadFile = File(...),
    subcategory: str = Form('bag')
):
    try:
        p_bytes = await person_image.read()
        a_bytes = await accessory_image.read()
        person_pil = Image.open(io.BytesIO(p_bytes)).convert('RGB')
        acc_pil = Image.open(io.BytesIO(a_bytes)).convert('RGB')
        w, h = person_pil.size
        sub = (subcategory or 'bag').lower()

        mask = Image.new('L', (w, h), 0)
        draw = ImageDraw.Draw(mask)

        if 'hat' in sub or 'cap' in sub:
            draw.rectangle([int(w * 0.20), int(h * 0.05), int(w * 0.80), int(h * 0.22)], fill=255)
            prompt = 'fashionable stylish hat on head, high fashion editorial'
        elif 'shoe' in sub or 'footwear' in sub or 'sneaker' in sub:
            draw.rectangle([int(w * 0.20), int(h * 0.78), int(w * 0.80), int(h * 0.98)], fill=255)
            prompt = 'designer shoes sneakers footwear, photorealistic high detail'
        else:
            # Default: Handbag / tote / shoulder bag (hip / side area)
            draw.rectangle([int(w * 0.55), int(h * 0.45), int(w * 0.95), int(h * 0.78)], fill=255)
            prompt = 'luxury designer leather handbag bag, high fashion editorial'

        if 'ip_pipe' in globals() and ip_pipe is not None:
            print(f'👜 Running IP-Adapter diffusion for {sub}...')
            ip_pipe.to('cuda')
            with torch.inference_mode():
                res = ip_pipe(
                    prompt=prompt,
                    image=person_pil.resize((512, 512)),
                    mask_image=mask.resize((512, 512)),
                    ip_adapter_image=acc_pil.resize((224, 224)),
                    num_inference_steps=25,
                    guidance_scale=7.0
                ).images[0]
            ip_pipe.to('cpu')
            torch.cuda.empty_cache()
            out_img = res.resize((w, h))
            print(f'✅ IP-Adapter accessory try-on completed for {sub}!')
        else:
            print(f'⚠️ IP-Adapter offline: returning baseline image for {sub}')
            out_img = person_pil

        buf = io.BytesIO()
        out_img.save(buf, format='PNG')
        buf.seek(0)
        return StreamingResponse(buf, media_type='image/png', headers={'X-TryOn-Studio': 'Accessory-Inpainting'})
    except Exception as e:
        import traceback
        traceback.print_exc()
        if 'ip_pipe' in globals() and ip_pipe is not None:
            try:
                ip_pipe.to('cpu')
            except Exception:
                pass
            torch.cuda.empty_cache()
        raise HTTPException(status_code=500, detail=str(e))

# ─── 4. Hairstyle Inpainting Studio ───────────────────────────────────────────
@app.post('/tryon-hair')
async def tryon_hair(
    person_image: UploadFile = File(...),
    hair_image: UploadFile = File(...),
    hairstyle_name: str = Form('hairstyle'),
    color_hex: str = Form('')
):
    try:
        p_bytes = await person_image.read()
        h_bytes = await hair_image.read()
        person_pil = Image.open(io.BytesIO(p_bytes)).convert('RGB')
        hair_pil = Image.open(io.BytesIO(h_bytes)).convert('RGB')
        w, h = person_pil.size
        h_name = (hairstyle_name or 'hairstyle').lower()

        mask = Image.new('L', (w, h), 0)
        draw = ImageDraw.Draw(mask)
        draw.rectangle([int(w * 0.12), 0, int(w * 0.88), int(h * 0.42)], fill=255)

        color_desc = f', dyed {color_hex} hair color' if color_hex else ''
        prompt = f'high quality photorealistic portrait, gorgeous {h_name} hairstyle{color_desc}, natural realistic hair strands, highly detailed, 8k'

        if 'ip_pipe' in globals() and ip_pipe is not None:
            print(f'💇 Running IP-Adapter diffusion for {h_name} hairstyle...')
            ip_pipe.to('cuda')
            with torch.inference_mode():
                res = ip_pipe(
                    prompt=prompt,
                    image=person_pil.resize((512, 512)),
                    mask_image=mask.resize((512, 512)),
                    ip_adapter_image=hair_pil.resize((224, 224)),
                    num_inference_steps=25,
                    guidance_scale=7.0
                ).images[0]
            ip_pipe.to('cpu')
            torch.cuda.empty_cache()
            out_img = res.resize((w, h))
            print(f'✅ IP-Adapter hairstyle try-on completed for {h_name}!')
        else:
            print(f'⚠️ IP-Adapter offline: returning baseline image for hairstyle')
            out_img = person_pil

        buf = io.BytesIO()
        out_img.save(buf, format='PNG')
        buf.seek(0)
        return StreamingResponse(buf, media_type='image/png', headers={'X-TryOn-Studio': 'IP-Adapter-Hairstyle'})
    except Exception as e:
        import traceback
        traceback.print_exc()
        if 'ip_pipe' in globals() and ip_pipe is not None:
            try:
                ip_pipe.to('cpu')
            except Exception:
                pass
            torch.cuda.empty_cache()
        raise HTTPException(status_code=500, detail=str(e))

# ─── Server Startup ──────────────────────────────────────────────────────────
def get_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(('127.0.0.1', 0))
        return s.getsockname()[1]

port = get_free_port()
print(f'🚀 Starting FastAPI server on port {port}...')
subprocess.run(['pkill', '-9', '-f', 'cloudflared'], stderr=subprocess.DEVNULL)
time.sleep(1)

def run_server():
    import asyncio
    asyncio.set_event_loop_policy(asyncio.DefaultEventLoopPolicy())
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host='127.0.0.1', port=port, log_level='warning', loop='asyncio')
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)

work_dir = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
log_path = os.path.join(work_dir, 'cloudflared.log')
if os.path.exists(log_path): os.remove(log_path)

log_file = open(log_path, 'w')
tunnel_proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}'], stdout=log_file, stderr=log_file)

tunnel_url = None
start_time = time.time()
while time.time() - start_time < 35:
    time.sleep(1)
    if os.path.exists(log_path):
        with open(log_path, 'r') as f:
            content = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if match:
                tunnel_url = match.group(0)
                break

if tunnel_url:
    print('\n' + '='*75)
    print('🎉 STYLEVISION UNIFIED MULTI-STUDIO GPU SERVER IS RUNNING!')
    print(f'👉 PUBLIC URL: {tunnel_url}')
    print('='*75)
    print('👗 Clothing: CatVTON (Diffusion Wardrobe Try-On)')
    print('💎 Jewelry: IP-Adapter Inpainting')
    print('👜 Accessories: Precision Inpainting')
    print('💇 Hairstyles: IP-Adapter Diffusion')
    print('🟢 Server is active. KEEP THIS CELL RUNNING!')
    try:
        while True: time.sleep(10)
    except KeyboardInterrupt:
        tunnel_proc.terminate()
else:
    print('❌ Cloudflare Tunnel timed out. Check /content/cloudflared.log')
